# ai-learn-18: RAG chunking strategies

Chunk long manuals five ways, retrieve with BM25, and check with character spans whether the answer survived. Everything is seeded (42) and runs in well under a second.

In [ ]:
import sys; sys.path.insert(0, '..')
from data import build_corpus
from chunkers import STRATEGIES, sentences, fit_lsa_vectorizer, n_words
from evaluate import build_index, evaluate, index_stats
docs, queries = build_corpus(42)
print(len(docs), 'docs,', len(queries), 'questions')
print(docs[0].text[:600])

## 1. Where does each chunker cut?
We print the first few chunk boundaries of manual 0 with `max_words=40`.

In [ ]:
vec = fit_lsa_vectorizer([d.text[s:e] for d in docs for s, e in sentences(d.text)], 16)
kw = {'fixed': {'size': 40}, 'overlap': {'size': 40, 'overlap': 13}, 'sentence': {'max_words': 40},
      'recursive': {'max_words': 40}, 'semantic': {'max_words': 40, 'min_words': 15, 'pct': 30, 'vectorize': vec}}
t = docs[0].text
for name, fn in STRATEGIES.items():
    spans = fn(t, **kw[name])
    print(f'--- {name}: {len(spans)} chunks')
    for s, e in spans[:3]:
        print('   |', t[s:e].replace(chr(10), ' / ')[:110])

## 2. A fact cut by a fixed window
A *hit* overlaps the fact sentence; *containment* needs the whole answer span inside one chunk.

In [ ]:
f = docs[0].facts
for ft, info in f.items():
    a0, a1 = info['answer_span']
    for s, e in STRATEGIES['fixed'](t, size=40):
        if s < a1 and e > a0 and not (s <= a0 and e >= a1):
            print(f'{ft}: answer {info["answer"]!r} is split at a fixed-40 boundary ->', repr(t[s:e][-60:]))

## 3. Evaluate every strategy, plain vs title-prefixed

In [ ]:
for prefix in (False, True):
    print('title_prefix =', prefix)
    for name, fn in STRATEGIES.items():
        k = dict(kw[name]); k.update({'size': 60} if 'size' in k else {'max_words': 60})
        if name == 'overlap': k['overlap'] = 20
        ch = build_index(docs, fn, title_prefix=prefix, **k)
        r = evaluate(docs, queries, ch); st = index_stats(docs, ch)
        print(f"  {name:9s} hit@3={r['hit@3']:.3f} contain@3={r['contain@3']:.3f} ctx@3={r['ctx_tokens@3']:.0f} chunks={st['n_chunks']} overhead={st['overhead_x']:.2f}x")

## 4. Full smoke run
Regenerates `results/` (tables, sweep and SVG plots).

In [ ]:
import os
os.chdir('..')
import run_smoke
m = run_smoke.main()
print(open('results/RESULTS.md').read()[:1500])